# YOLO26-S Drop-Backbone v2 — RSNA Pneumonia Detection

## Ablation Study: Removing Early Backbone Block (model.2 C3k2 at 160x160)

**Everything identical to baseline v3:**
- Dataset: yolo_dataset_v3 (No Not Normal class)
- imgsz=1024, batch=4, optimizer=AdamW, lr0=0.001, patience=30

**Only change: model architecture**
- Removed: model.2 (C3k2 at 160x160 resolution)


## Step 1 — Imports & Paths

In [ ]:
import os, shutil, warnings, time, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from tqdm import tqdm
import torch
warnings.filterwarnings('ignore')
print('All imports successful')

## Step 2 — Paths
Identical to baseline. Reuses same dataset, same YOLO_DIR.

In [ ]:
BASE_DIR    = Path(r'C:\Users\chait\Documents\rsna_pneumonia')
YOLO_DIR    = BASE_DIR / 'yolo_dataset_v3'     # v3 clean dataset
RUNS_DIR    = BASE_DIR / 'runs'
PLOTS_DIR   = BASE_DIR / 'paper_plots_v3'      # v3 plots
MODELS_DIR  = BASE_DIR / 'models'
CONFIG_DIR  = BASE_DIR / 'configs'

for d in [RUNS_DIR, PLOTS_DIR, MODELS_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

yaml_path = YOLO_DIR / 'data.yaml'

n_train = len(list((YOLO_DIR/'images'/'train').glob('*.png')))
n_val   = len(list((YOLO_DIR/'images'/'val').glob('*.png')))

assert n_train > 0, 'Train images not found — run baseline v3 notebook first'
assert n_val   > 0, 'Val images not found — run baseline v3 notebook first'
assert yaml_path.exists(), 'data.yaml not found — run baseline v3 notebook first'

print('Paths ready')
print(f'  YOLO dir : {YOLO_DIR} (v3 clean dataset)')
print(f'  Train    : {n_train} images')
print(f'  Val      : {n_val} images')


## Step 3 — Create Modified YAML Architecture Config

Removes model.2 (C3k2 at 160x160). All layer indices shift by -1 from model.3 onwards.

Verified spatial dimensions:
- Concat at model.11: [-1]=40x40 + model.5=40x40 PASS
- Concat at model.14: [-1]=20x20 + model.7=20x20 PASS
- Detect inputs: model.12=40x40 + model.15=20x20 PASS

In [ ]:
dropbackbone_yaml_str = """
# YOLO26-S Drop-Backbone Variant
# Ablation: Removed model.2 (C3k2 at 160x160 resolution)
# Justified: early low-level features at 160x160 are redundant for pneumonia opacity detection
# Neck references updated: P3=model.5, P4=model.7, P5=model.9
# Spatial dimensions verified at all concat points

nc: 1
scales:
  s: [0.50, 0.50, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]           # 0  640->320
  - [-1, 1, Conv, [128, 3, 2]]          # 1  320->160
  # model.2 C3k2 DROPPED — early low-level feature block at 160x160
  - [-1, 1, Conv, [256, 3, 2]]          # 2  160->80   (was model.3)
  - [-1, 2, C3k2, [512, False, 0.25]]   # 3  80->80    (was model.4)
  - [-1, 1, Conv, [512, 3, 2]]          # 4  80->40    (was model.5)
  - [-1, 2, C3k2, [512, True]]          # 5  40->40    (was model.6) P3
  - [-1, 1, Conv, [1024, 3, 2]]         # 6  40->20    (was model.7)
  - [-1, 2, C3k2, [1024, True]]         # 7  20->20    (was model.8) P4
  - [-1, 1, SPPF, [1024, 5]]            # 8  20->20    (was model.9)
  - [-1, 1, C2PSA, [1024]]              # 9  20->20    (was model.10) P5

head:
  # Upsample P5 (20x20) -> 40x40, concat with P3 backbone model.5 (40x40)
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]   # 10  20->40
  - [[-1, 5], 1, Concat, [1]]                    # 11  concat with model.5 P3=40x40
  - [-1, 2, C3k2, [512, False]]                  # 12  40x40 fused features

  # Downsample 40x40 -> 20x20, concat with P4 backbone model.7 (20x20)
  - [-1, 1, Conv, [512, 3, 2]]                   # 13  40->20
  - [[-1, 7], 1, Concat, [1]]                    # 14  concat with model.7 P4=20x20
  - [-1, 2, C3k2, [1024, False]]                 # 15  20x20 fused features

  # Detect on 2 scales: model.12 (40x40) and model.15 (20x20)
  - [[12, 15], 1, Detect, [nc]]                  # 16  detection head
"""

config_path = CONFIG_DIR / 'yolo26s_dropbackbone.yaml'
with open(config_path, 'w') as f:
    f.write(dropbackbone_yaml_str)

print(f'Drop-backbone config saved: {config_path}')
print(open(config_path).read())

## Step 4 — Verify Architecture & Parameter Count

In [ ]:
from ultralytics import YOLO

print('Loading drop-backbone model...')
dropbackbone_model = YOLO(str(config_path))
dropbackbone_params = sum(p.numel() for p in dropbackbone_model.model.parameters())

print('Loading baseline for comparison...')
baseline_model = YOLO('yolo26s.pt')
baseline_params = sum(p.numel() for p in baseline_model.model.parameters())
del baseline_model
torch.cuda.empty_cache()

reduction = (baseline_params - dropbackbone_params) / baseline_params * 100

print(f'\nParameter comparison:')
print(f'  Baseline       : {baseline_params/1e6:.2f}M')
print(f'  Drop-backbone  : {dropbackbone_params/1e6:.2f}M')
print(f'  Reduction      : {reduction:.1f}%')

print(f'\nDrop-backbone architecture layers:')
for name, module in dropbackbone_model.model.named_modules():
    parts = name.split('.')
    if len(parts) == 2 and parts[0] == 'model':
        try:
            idx = int(parts[1])
            print(f'  model.{idx:2d}  {type(module).__name__}')
        except:
            pass

detect_layer = dropbackbone_model.model.model[-1]
print(f'\nDetection heads : {detect_layer.nl}')
print(f'Strides         : {detect_layer.stride}')

del dropbackbone_model
torch.cuda.empty_cache()
print('\nVerification complete')

## Step 5 — Train Drop-Backbone Model

Training config IDENTICAL to baseline. Only name and model architecture differ.

In [ ]:
from ultralytics import YOLO

model = YOLO(str(config_path))
print('Drop-backbone model loaded')
model.info()

In [ ]:
TRAIN_CONFIG = dict(
    data            = str(yaml_path),
    epochs          = 75,
    imgsz           = 1024,
    batch           = 4,
    device          = 0,
    patience        = 30,
    project         = str(RUNS_DIR),
    name            = 'yolo26s_dropbackbone_v2',
    exist_ok        = True,
    pretrained      = True,
    optimizer       = 'AdamW',
    lr0             = 0.001,
    lrf             = 0.01,
    weight_decay    = 0.0005,
    warmup_epochs   = 3.0,
    warmup_momentum = 0.8,
    warmup_bias_lr  = 0.1,
    seed            = 0,
    deterministic   = True,
    verbose         = True,
    save            = True,
    save_period     = 10,
    plots           = True,
    hsv_h           = 0.0,
    hsv_s           = 0.0,
    hsv_v           = 0.2,
    degrees         = 10,
    translate       = 0.05,
    scale           = 0.05,
    shear           = 0.0,
    perspective     = 0.0,
    flipud          = 0.0,
    fliplr          = 0.5,
    mosaic          = 0.0,
    mixup           = 0.0,
    copy_paste      = 0.0,
    close_mosaic    = 10,
    amp             = True,
    box             = 7.5,
    cls             = 0.5,
    dfl             = 1.5,
    nbs             = 64,
)
print('Config ready')
for k,v in TRAIN_CONFIG.items(): print(f'   {k:20s}: {v}')


In [ ]:
print('='*60)
print('  YOLO26-S DROP-BACKBONE v2 — TRAINING STARTED')
print('  Ablation: Removed model.2 (C3k2 at 160x160)')
print('  Dataset: v3 clean (no Not Normal), imgsz=1024, AdamW')
print('='*60)

results = model.train(**TRAIN_CONFIG)

print('='*60)
print('  TRAINING COMPLETE')
print('='*60)


## Step 6 — Load Best Weights & Validate

In [ ]:
best_weights = RUNS_DIR / 'yolo26s_dropbackbone_v2' / 'weights' / 'best.pt'
print(f'Loading: {best_weights}')
best_model  = YOLO(str(best_weights))
val_metrics = best_model.val(data=str(yaml_path), imgsz=1024, device=0)
print('Validation complete')


## Step 7 — Extract & Display All Metrics

In [ ]:
map50   = val_metrics.box.map50
map75   = val_metrics.box.map75
map5095 = val_metrics.box.map
prec    = val_metrics.box.mp
rec     = val_metrics.box.mr
f1      = 2*(prec*rec)/(prec+rec+1e-8)
nparams = sum(p.numel() for p in best_model.model.parameters())/1e6

val_imgs_list = list((YOLO_DIR/'images'/'val').glob('*.png'))[:100]
t0  = time.time()
for ip in val_imgs_list: best_model.predict(str(ip), imgsz=1024, device=0, verbose=False)
fps = len(val_imgs_list)/(time.time()-t0)

print('\n'+'='*50)
print('  YOLO26-S DROP-BACKBONE v2 — FINAL RESULTS')
print('='*50)
for name, val in [('mAP@0.5',map50),('mAP@0.75',map75),('mAP@0.5:0.95',map5095),
                  ('Precision',prec),('Recall',rec),('F1 Score',f1),
                  ('Params (M)',nparams),('FPS',fps)]:
    print(f'  {name:18s}: {val:.4f}')
print('='*50)

new_row = pd.DataFrame([{
    'Model'        : 'YOLO26-S Drop-Backbone v2',
    'mAP@0.5'      : round(map50,4),
    'mAP@0.75'     : round(map75,4),
    'mAP@0.5:0.95' : round(map5095,4),
    'Precision'    : round(prec,4),
    'Recall'       : round(rec,4),
    'F1'           : round(f1,4),
    'Params(M)'    : round(nparams,2),
    'FPS'          : round(fps,1)
}])

csv_path = BASE_DIR / 'results_table_v3.csv'
if csv_path.exists():
    existing = pd.read_csv(csv_path)
    existing = existing[existing['Model'] != 'YOLO26-S Drop-Backbone v2']
    combined = pd.concat([existing, new_row], ignore_index=True)
else:
    combined = new_row

combined.to_csv(csv_path, index=False)
print(f'\nResults appended to: {csv_path}')
print('\nFull ablation results:')
print(combined.to_string(index=False))

## Step 8 — Training Curve Plots

In [ ]:
df_res = pd.read_csv(RUNS_DIR/'yolo26s_dropbackbone_v2'/'results.csv')
df_res.columns = df_res.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(18,5))
fig.suptitle('YOLO26-S Drop-Backbone — Training & Validation Loss',
             fontsize=14, fontweight='bold')
for ax, (tc,vc,title) in zip(axes,[
    ('train/box_loss','val/box_loss','Box Loss'),
    ('train/cls_loss','val/cls_loss','Classification Loss'),
    ('train/dfl_loss','val/dfl_loss','DFL Loss')]):
    if tc in df_res.columns: ax.plot(df_res['epoch'],df_res[tc],label='Train',color='#3498db',lw=2)
    if vc in df_res.columns: ax.plot(df_res['epoch'],df_res[vc],label='Val',color='#e74c3c',lw=2,ls='--')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR/'fig15_dropbackbone_loss_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig15_dropbackbone_loss_curves.png')

fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('YOLO26-S Drop-Backbone — mAP Progression',
             fontsize=14, fontweight='bold')
for ax, (col,label,color) in zip(axes,[
    ('metrics/mAP50(B)',   'mAP@0.5',      '#2ecc71'),
    ('metrics/mAP50-95(B)','mAP@0.5:0.95','#9b59b6')]):
    if col in df_res.columns:
        ax.plot(df_res['epoch'],df_res[col],color=color,lw=2)
        bi = df_res[col].idxmax()
        ax.axvline(df_res.loc[bi,'epoch'],color='red',ls='--',
                   alpha=0.7, label=f"Best: {df_res[col].max():.4f}")
    ax.set_title(label); ax.set_xlabel('Epoch'); ax.set_ylabel(label)
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR/'fig16_dropbackbone_map_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig16_dropbackbone_map_curves.png')

## Step 9 — Copy Auto-Generated YOLO Plots

In [ ]:
run_dir = RUNS_DIR / 'yolo26s_dropbackbone_v2'
for plot in ['confusion_matrix.png','confusion_matrix_normalized.png',
             'PR_curve.png','P_curve.png','R_curve.png','F1_curve.png','results.png']:
    src = run_dir / plot
    if src.exists():
        shutil.copy2(str(src), str(PLOTS_DIR/f'dropbackbone_{plot}'))
        print(f'Copied: {plot}')
    else:
        print(f'Not found: {plot}')

## Step 10 — Sample Predictions

In [ ]:
val_samples = list((YOLO_DIR/'images'/'val').glob('*.png'))[:8]
fig, axes   = plt.subplots(2, 4, figsize=(20,10))
fig.suptitle('YOLO26-S Drop-Backbone — Sample Predictions on Validation Set',
             fontsize=14, fontweight='bold')
for ax, ip in zip(axes.flatten(), val_samples):
    res = best_model.predict(str(ip), imgsz=1024, device=0, conf=0.25, verbose=False)[0]
    ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(ip.stem[:18], fontsize=8); ax.axis('off')
plt.tight_layout()
plt.savefig(PLOTS_DIR/'fig17_dropbackbone_sample_predictions.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig17_dropbackbone_sample_predictions.png')

## Step 11 — Save Model & Final Ablation Summary

In [ ]:
final_path = MODELS_DIR / 'yolo26s_dropbackbone_v2_best.pt'
shutil.copy2(str(best_weights), str(final_path))

print('Model saved:', final_path)
print('\n'+'='*60)
print('  FULL ABLATION STUDY COMPLETE')
print('='*60)
print('\nFinal results table:')
print(pd.read_csv(BASE_DIR/'results_table_v3.csv').to_string(index=False))
print('\nPaper plots generated:')
for f in sorted(PLOTS_DIR.glob('*.png')): print(f'   {f.name}')